# 1. Extract

Read the CRM and ERP CSV files from Azure Blob Storage into a dictionary of raw DataFrames.

In [106]:
import os
import re
from io import BytesIO

import pandas as pd
from azure.storage.blob import BlobServiceClient
from dotenv import load_dotenv
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL


load_dotenv()

AZURE_CONTAINER = "baraa"
AZURE_CONNECTION_STRING = os.getenv("AZURE_STORAGE_CONNECTION_STRING")


def read_blob_csv(container_name, blob_path, connection_string):
    """Read one CSV file from Azure Blob Storage."""
    if not connection_string:
        raise ValueError("AZURE_STORAGE_CONNECTION_STRING is not set.")

    blob_service = BlobServiceClient.from_connection_string(connection_string)
    blob_client = blob_service.get_blob_client(
        container=container_name,
        blob=blob_path,
    )
    blob_data = blob_client.download_blob().readall()
    return pd.read_csv(BytesIO(blob_data))


## Source Files

**Azure container:** `baraa`

### CRM

- `source_crm/cust_info.csv`
- `source_crm/prd_info.csv`
- `source_crm/sales_details.csv`

### ERP

- `source_erp/CUST_AZ12.csv`
- `source_erp/LOC_A101.csv`
- `source_erp/PX_CAT_G1V2.csv`

In [107]:
def extract_data(container_name=AZURE_CONTAINER):
    """Extract all CRM and ERP source files into a dictionary."""
    source_files = {
        "crm_customer": "source_crm/cust_info.csv",
        "crm_product": "source_crm/prd_info.csv",
        "crm_sales": "source_crm/sales_details.csv",
        "erp_customer": "source_erp/CUST_AZ12.csv",
        "erp_location": "source_erp/LOC_A101.csv",
        "erp_category": "source_erp/PX_CAT_G1V2.csv",
    }

    extracted_data = {
        name: read_blob_csv(
            container_name,
            blob_path,
            AZURE_CONNECTION_STRING,
        )
        for name, blob_path in source_files.items()
    }

    print(f"Extracted {len(extracted_data)} source files.")
    return extracted_data


raw_data = extract_data()


Extracted 6 source files.


# 2. Transform

Clean source-specific data, apply business rules, normalize column names, and convert date fields to date-only values.

The stage is orchestrated by `transform_data(raw_data)` and returns the cleaned DataFrames in a dictionary.

In [108]:
def clean_customer_data(dataframe):
    """Clean CRM customer master data."""
    df = dataframe.copy().drop_duplicates()
    df = df.dropna(subset=["cst_id", "cst_key"])
    df["cst_id"] = pd.to_numeric(df["cst_id"], errors="coerce")
    df = df.dropna(subset=["cst_id"])
    df["cst_id"] = df["cst_id"].astype(int)
    df["cst_create_date"] = pd.to_datetime(
        df["cst_create_date"], errors="coerce"
    )
    df = (
        df.sort_values("cst_create_date")
        .drop_duplicates("cst_id", keep="last")
    )

    name_columns = ["cst_firstname2", "cst_lastname2"]
    df[name_columns] = df[name_columns].apply(
        lambda column: column.astype("string").str.strip()
    )
    df = df.rename(columns={
        "cst_firstname2": "cst_firstname",
        "cst_lastname2": "cst_lastname",
    })
    df["cst_marital_status"] = df["cst_marital_status"].map({
        "M": "Married",
        "S": "Single",
    }).fillna("N/A")
    df["cst_gndr"] = df["cst_gndr"].map({
        "M": "Male",
        "F": "Female",
    }).fillna("N/A")
    return df


def clean_product_data(dataframe):
    """Clean CRM product master data and repair date ranges."""
    df = dataframe.copy().drop_duplicates()
    df = df.dropna(subset=["prd_id", "prd_key"])

    df["cat_id"] = df["prd_key"].str[:5].str.replace("-", "_")
    df["prd_key"] = df["prd_key"].str[6:]
    category_id = df.pop("cat_id")
    df.insert(df.columns.get_loc("prd_id") + 1, "cat_id", category_id)

    df["prd_cost"] = pd.to_numeric(df["prd_cost"], errors="coerce").fillna(0)
    df["prd_line"] = df["prd_line"].astype("string").str.strip().str.upper().map({
        "M": "Mountain",
        "R": "Road",
        "S": "Other Sale",
        "T": "Touring",
    }).fillna("N/A")
    df["prd_start_dt"] = pd.to_datetime(df["prd_start_dt"], errors="coerce")
    df["prd_end_dt"] = pd.to_datetime(df["prd_end_dt"], errors="coerce")
    df = df.sort_values(["prd_key", "prd_start_dt"])

    broken_dates = df["prd_start_dt"] > df["prd_end_dt"]
    next_start = (
        df.groupby("prd_key")["prd_start_dt"].shift(-1)
        - pd.Timedelta(days=1)
    )
    df.loc[broken_dates, "prd_end_dt"] = next_start[broken_dates]
    return df


def clean_sales_data(dataframe):
    """Clean CRM sales transactions and recalculate sales."""
    df = dataframe.copy()

    order_number = df["sls_ord_num"].astype("string").str.strip()
    order_id = order_number.str.extract(r"^SO(\d{5})$", expand=False)
    valid_order = order_id.notna() & pd.to_numeric(
        order_id, errors="coerce"
    ).gt(0)
    df.loc[~valid_order, "sls_ord_num"] = pd.NA

    for column in ["sls_order_dt", "sls_ship_dt", "sls_due_dt"]:
        parsed_date = pd.to_datetime(
            df[column].astype("string"),
            format="%Y%m%d",
            errors="coerce",
        )
        valid_year = parsed_date.dt.year.between(1900, 2050)
        df[column] = parsed_date.where(valid_year)

    quantity = pd.to_numeric(df["sls_quantity"], errors="coerce")
    valid_quantity = quantity.gt(0) & quantity.mod(1).eq(0)
    df["sls_quantity"] = quantity.where(valid_quantity, 0).astype("Int64")
    df["sls_price"] = pd.to_numeric(
        df["sls_price"], errors="coerce"
    ).abs()
    df["sls_sales"] = df["sls_quantity"] * df["sls_price"]
    return df


def clean_erp_customer_data(dataframe):
    """Clean ERP customer demographics."""
    df = dataframe.copy()
    df["CID"] = df["CID"].astype("string").str.replace(
        r"^NAS", "", regex=True
    )
    df["BDATE"] = pd.to_datetime(df["BDATE"], errors="coerce")
    df.loc[df["BDATE"] >= pd.Timestamp.now(), "BDATE"] = pd.NaT
    gender = df["GEN"].astype("string").str.strip().str.upper()
    df["GEN"] = gender.map({
        "M": "Male",
        "MALE": "Male",
        "F": "Female",
        "FEMALE": "Female",
    }).fillna("N/A")
    return df


def clean_erp_location_data(dataframe):
    """Clean ERP customer location data."""
    df = dataframe.copy()
    df["CID"] = df["CID"].astype("string").str.strip().str.replace(
        "-", "_", regex=False
    )
    country = df["CNTRY"].astype("string").str.strip()
    country_code = country.str.upper()
    df["CNTRY"] = country.fillna("N/A").replace("", "N/A")
    df.loc[country_code.eq("DE"), "CNTRY"] = "Germany"
    df.loc[country_code.isin(["US", "USA"]), "CNTRY"] = "United State"
    return df


def to_snake_case(column_name):
    """Convert a column name to snake_case, preserving acronyms."""
    name = str(column_name).strip()
    name = re.sub(r"([A-Z]+)([A-Z][a-z])", r"\1_\2", name)
    name = re.sub(r"([a-z0-9])([A-Z])", r"\1_\2", name)
    name = re.sub(r"[^a-zA-Z0-9]+", "_", name)
    return name.strip("_").lower()


def normalize_dataframe(dataframe):
    """Standardize column names and convert date fields to date-only values."""
    df = dataframe.copy()
    df.columns = [to_snake_case(column) for column in df.columns]
    date_columns = [
        column for column in df.columns
        if "date" in column or column.endswith("_dt") or column == "bdate"
    ]
    for column in date_columns:
        parsed_date = pd.to_datetime(df[column], errors="coerce")
        df[column] = parsed_date.dt.date
    return df


In [109]:
def transform_data(raw_data):
    """Transform all extracted sources and return cleaned DataFrames."""
    transformed_data = {
        "df_customer": clean_customer_data(raw_data["crm_customer"]),
        "df_product": clean_product_data(raw_data["crm_product"]),
        "df_sales": clean_sales_data(raw_data["crm_sales"]),
        "df_customer_erp": clean_erp_customer_data(raw_data["erp_customer"]),
        "df_location": clean_erp_location_data(raw_data["erp_location"]),
        "df_category": raw_data["erp_category"].copy(),
    }
    return {
        name: normalize_dataframe(dataframe)
        for name, dataframe in transformed_data.items()
    }


cleaned_data = transform_data(raw_data)

df_customer = cleaned_data["df_customer"]
df_product = cleaned_data["df_product"]
df_sales = cleaned_data["df_sales"]
df_customer_erp = cleaned_data["df_customer_erp"]
df_location = cleaned_data["df_location"]
df_category = cleaned_data["df_category"]

print("Transformation completed for all datasets.")


Transformation completed for all datasets.


# Validate

Run focused quality checks after transformation and before loading.

In [110]:
def validate_data(cleaned_data):
    """Run quality checks and return a validation report."""
    sales = cleaned_data["df_sales"]
    product = cleaned_data["df_product"]
    erp_customer = cleaned_data["df_customer_erp"]
    location = cleaned_data["df_location"]

    report = {
        "sales_invalid_order_numbers": sales["sls_ord_num"].isna().sum(),
        "sales_date_order_errors": (
            (sales["sls_order_dt"] > sales["sls_ship_dt"])
            | (sales["sls_order_dt"] > sales["sls_due_dt"])
        ).sum(),
        "sales_invalid_quantities": (sales["sls_quantity"] < 0).sum(),
        "sales_negative_prices": (sales["sls_price"] < 0).sum(),
        "product_date_range_errors": (
            product["prd_start_dt"] > product["prd_end_dt"]
        ).sum(),
        "erp_future_birth_dates": (
            erp_customer["bdate"] >= pd.Timestamp.now().date()
        ).sum(),
        "allowed_gender_values": sorted(erp_customer["gen"].unique().tolist()),
        "location_null_countries": (location["cntry"] == "N/A").sum(),
    }

    print("Validation report:")
    for check, result in report.items():
        print(f"- {check}: {result}")
    return report


validation_report = validate_data(cleaned_data)


Validation report:
- sales_invalid_order_numbers: 0
- sales_date_order_errors: 0
- sales_invalid_quantities: 0
- sales_negative_prices: 0
- product_date_range_errors: 0
- erp_future_birth_dates: 0
- allowed_gender_values: ['Female', 'Male', 'N/A']
- location_null_countries: 337


# Load

Load the validated DataFrames into the MySQL `baraa_v2` schema.

In [111]:
def load_data(cleaned_data, database_name="baraa_v2"):
    """Create the target schema and load cleaned DataFrames into MySQL."""
    load_dotenv()
    mysql_host = os.getenv("MYSQL_HOST")
    mysql_user = os.getenv("MYSQL_USER")
    mysql_password = os.getenv("MYSQL_PASSWORD")
    mysql_port = int(os.getenv("MYSQL_PORT", "3306"))

    missing_settings = [
        name
        for name, value in {
            "MYSQL_HOST": mysql_host,
            "MYSQL_USER": mysql_user,
            "MYSQL_PASSWORD": mysql_password,
        }.items()
        if not value
    ]
    if missing_settings:
        print("MySQL load skipped. Missing: " + ", ".join(missing_settings))
        return None

    server_url = URL.create(
        drivername="mysql+pymysql",
        username=mysql_user,
        password=mysql_password,
        host=mysql_host,
        port=mysql_port,
    )
    server_engine = create_engine(server_url)

    # Create the schema separately because it may not exist yet.
    with server_engine.begin() as connection:
        connection.execute(
            text(f"CREATE DATABASE IF NOT EXISTS `{database_name}`")
        )

    engine = create_engine(server_url.set(database=database_name))
    table_map = {
        "df_customer": "dim_customer",
        "df_product": "dim_product",
        "df_sales": "fact_sales",
        "df_customer_erp": "dim_customer_erp",
        "df_location": "dim_location",
        "df_category": "dim_category",
    }

    with engine.begin() as connection:
        connection.execute(text("SET FOREIGN_KEY_CHECKS = 0"))
        for dataframe_name, table_name in table_map.items():
            connection.execute(text(f"DROP TABLE IF EXISTS `{table_name}`"))
            cleaned_data[dataframe_name].to_sql(
                table_name,
                connection,
                if_exists="append",
                index=False,
            )
        connection.execute(text("SET FOREIGN_KEY_CHECKS = 1"))

    loaded_tables = pd.read_sql(text("SHOW TABLES"), engine)
    print(f"Loaded {len(table_map)} tables into {database_name}.")
    print(loaded_tables.to_string(index=False))
    return engine


In [112]:
engine = load_data(cleaned_data)

Loaded 6 tables into baraa_v2.
Tables_in_baraa_v2
      dim_category
      dim_customer
  dim_customer_erp
      dim_location
       dim_product
        fact_sales
